# Labor Market — Research Workbench

Research mirror of `sections/labor.py`. Uses the exact same code path as the Streamlit app — no duplicated logic.

Run the **Setup** cells once per Colab session, then jump to whichever section you want.

## Setup (standard 5 cells — same for every section notebook)

In [ ]:
!pip install fredapi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import os
PROJECT_PATH = "/content/drive/MyDrive/FREDMACRO"

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

In [ ]:
os.environ["FRED_API_KEY"] = "6e079bc3e1ab2b8280b94e05ff432f30"

In [ ]:
# Sanity: confirm the shared modules import cleanly
from core import config, transforms
from core.fred_client import get_series
from sections import labor

print('config.DEFAULT_START_DATE =', config.DEFAULT_START_DATE)
print('config.CACHE_DIR =', config.CACHE_DIR)

## Pattern A — Call individual chart functions

Each chart is its own function in `sections.labor`. They all return `plotly.graph_objects.Figure` and display with `.show()` in Colab, the same way Streamlit displays them with `st.plotly_chart()`.

In [ ]:
labor.nfp_overview(start_date='2022-01-01').show()

In [ ]:
labor.claims(start_date='2022-01-01').show()

In [ ]:
labor.jolts().show()

In [ ]:
labor.unemployment_and_jwg(start_date='2022-01-01').show()

In [ ]:
labor.nfp_sectors_grid(start_date='2022-01-01').show()

In [ ]:
# Diffusion index — try both variants
labor.diffusion_index_chart(extended=False).show()
labor.diffusion_index_chart(extended=True).show()

In [ ]:
# Cyclical view — both modes
labor.cyclical_view(view='contribution').show()
labor.cyclical_view(view='grid').show()

## Pattern B — Inspect raw data

The transform layer is pure — call it on FRED data and inspect the result without touching plots. Useful for sanity-checking that the math matches your mental model.

In [ ]:
# What does the headline NFP frame look like?
nfp = get_series('PAYEMS')
df = transforms.with_change_and_ma(nfp, 'NFP', start_date='2024-01-01')
df.tail(12)

In [ ]:
# Cyclical split — what's driving headline NFP?
from core.transforms import cyclical_split

split = cyclical_split(get_series('PAYEMS'), get_series('USGOVT'), get_series('USEHS'))
split.tail(6)

## Pattern C — Build the whole section like Streamlit does

`labor.build()` is what `app.py` calls. The output is a dict; iterate it just like the Streamlit app does.

In [ ]:
section = labor.build(
    start_date='2022-01-01',
    diffusion_extended=True,
    cyclical_view_mode='contribution',
)

print(section['title'])
for chart in section['charts']:
    print(f"  - {chart['id']}: {chart['title']}")

In [ ]:
# Show every chart in order, exactly as Streamlit would
for chart in section['charts']:
    print(chart['title'])
    chart['fig'].show()
    if chart.get('commentary'):
        print('   →', chart['commentary'])
    print()

## Cache control

FRED responses are cached on disk under `.cache/`. To force a fresh fetch:

In [ ]:
# Uncomment to wipe the on-disk FRED cache
# from core.fred_client import clear_cache
# clear_cache()